# 02 EDA And Feature Exploration

This notebook is for exploratory analysis and visualization only. Reusable business logic stays in `src/data` and `src/features` so the recommender system remains modular and production-readable.

## How These Features Feed The Recommenders

- The `Preprocessor` creates clean `track_level` and `artist_level` tables from raw Spotify payloads.
- The `FeatureBuilder` converts track-level rows into standardized numeric content features.
- The `SimilarityBuilder` computes cosine similarity across those standardized track vectors.
- Later, `ContentRecommender` can use these matrices for nearest-neighbor retrieval or candidate generation inside the hybrid stack.

In [ ]:
# Add the repository's src directory so notebook imports behave like local development.
from pathlib import Path
import sys

# Use pandas for tabular inspection and numpy for quick numeric summaries.
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from config.settings import ProjectSettings
from features.feature_builder import FeatureBuilder
from features.similarity_builder import SimilarityBuilder

settings = ProjectSettings.from_env()
settings.ensure_project_directories()

In [ ]:
# Find the most recent processed tables so EDA can start from the curated preprocessing layer.
processed_dir = settings.processed_data_dir

def load_latest_csv(dataset_suffix: str) -> pd.DataFrame:
    """Load the most recent processed CSV that matches a dataset suffix."""
    matching_files = sorted(processed_dir.glob(f'*_{dataset_suffix}.csv'))
    if not matching_files:
        return pd.DataFrame()
    return pd.read_csv(matching_files[-1])

track_level_df = load_latest_csv('track_level')
artist_level_df = load_latest_csv('artist_level')

track_level_df.head(), artist_level_df.head()

In [ ]:
# Inspect schema, missingness, and basic numeric summaries before feature engineering.
track_level_df.info()

# Missing-value inspection is useful for spotting sparse metadata fields.
track_level_df.isna().mean().sort_values(ascending=False).head(10)

In [ ]:
# Build standardized content features from the curated track-level table.
feature_builder = FeatureBuilder(max_genre_features=15)
feature_artifacts = feature_builder.create_model_ready_feature_matrix(track_level_df)

# These standardized features become the numeric input space for the content recommender.
print('Number of tracks:', len(feature_artifacts.track_ids))
print('Number of features:', len(feature_artifacts.feature_names))
feature_artifacts.feature_frame.head()

In [ ]:
# Compute cosine similarity so we can inspect nearest neighbors in content space.
similarity_builder = SimilarityBuilder(settings=settings)
similarity_artifacts = similarity_builder.compute_cosine_similarity(feature_artifacts)

# This exact output can later power track-to-track retrieval in a content recommender.
if feature_artifacts.track_ids:
    query_track_id = feature_artifacts.track_ids[0]
    neighbors = similarity_builder.get_top_k_similar_tracks(similarity_artifacts, query_track_id, k=5)
    print('Query track:', query_track_id)
    print('Nearest neighbors:', neighbors)
else:
    print('No processed track-level data found yet. Run the sample data collection CLI first.')

In [ ]:
# Optional visualization example:
# If matplotlib is installed in your local environment, this quick histogram is a simple first look.
# Uncomment the lines below when you want a lightweight visualization.
# import matplotlib.pyplot as plt
# track_level_df[['danceability', 'energy', 'valence', 'tempo']].hist(figsize=(10, 6))
# plt.tight_layout()
# plt.show()